In [3]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def crawl_naver_blog(keyword, max_posts=50):
    # 크롬 드라이버 설정
    options = webdriver.ChromeOptions()
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--headless') # 브라우저 창을 띄우지 않으려면 주석 해제 (단, 차단될 확률이 높아질 수 있음)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # 2024.01.01 ~ 2026.10.22 기간 필터가 적용된 네이버 블로그 검색 URL
    # nso=so:r,p:from20240101to20261022 부분으로 날짜 지정
    url = f"https://search.naver.com/search.naver?where=blog&query={keyword}&nso=so%3Ar%2Cp%3Afrom20240101to20261022"
    driver.get(url)
    time.sleep(2)
    
    post_links = []
    
    # 1. 블로그 게시글 링크 수집 (스크롤을 내리며 수집)
    print("게시글 링크를 수집 중입니다...")
    while len(post_links) < max_posts:
        # 현재 페이지의 블로그 링크들 찾기
        elements = driver.find_elements(By.CSS_SELECTOR, ".api_txt_lines.total_tit")
        for el in elements:
            link = el.get_attribute("href")
            if link and link not in post_links:
                post_links.append(link)
                if len(post_links) >= max_posts:
                    break
        
        # 스크롤을 최하단으로 내려 다음 게시글 로딩
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
        
        # 더 이상 로딩될 링크가 없을 경우 대비 (무한루프 방지)
        # 실제 구현시에는 이전 높이와 현재 높이를 비교하는 로직 추가 권장
    
    print(f"총 {len(post_links)}개의 링크 수집 완료. 상세 데이터 추출을 시작합니다.")
    
    # 2. 각 링크에 들어가서 상세 데이터 수집
    data_list = []
    
    for idx, link in enumerate(post_links, 1):
        try:
            driver.get(link)
            time.sleep(2)
            
            # 네이버 블로그는 본문이 iframe(mainFrame) 안에 존재함
            driver.switch_to.frame("mainFrame")
            
            # iframe 안에서 렌더링된 HTML 가져오기
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            
            # 데이터 초기화
            blogger_id = ""
            post_date = ""
            post_text = ""
            like_count = "0"
            comment_count = "0"
            tags = ""
            
            # 1) 아이디 수집 (URL에서 추출하거나 화면에서 추출)
            # URL 예시: https://blog.naver.com/userId/postNum
            blogger_id = link.split('/')[-2]
            
            # 2) 날짜 수집
            date_tag = soup.select_one('.se_publishDate') or soup.select_one('.blog2_date')
            if date_tag:
                post_date = date_tag.text.strip()
                
            # 3) 본문 텍스트 수집 (스마트에디터 ONE 기준)
            content_tag = soup.select_one('.se-main-container') or soup.select_one('#postViewArea')
            if content_tag:
                post_text = content_tag.get_text(separator=' ', strip=True)
                
            # 4) 태그 수집
            tag_elements = soup.select('.item_tag') or soup.select('.wrap_tag > a')
            tags = ", ".join([tag.text.strip().replace('#', '') for tag in tag_elements])
            
            # 5) 좋아요 수, 댓글 수 수집 (동적 로딩)
            # iframe 전환 후 일정 시간 대기해야 버튼에 숫자가 렌더링됨
            try:
                # 좋아요 수 (공감 버튼 내의 숫자)
                WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "em.u_cnt._count"))
                )
                like_count_element = driver.find_element(By.CSS_SELECTOR, "em.u_cnt._count")
                if like_count_element.text:
                    like_count = like_count_element.text
            except:
                like_count = "0" # 좋아요를 막아둔 경우
                
            try:
                # 댓글 수
                comment_count_element = driver.find_element(By.CSS_SELECTOR, "em#commentCount")
                if comment_count_element.text:
                    comment_count = comment_count_element.text
            except:
                comment_count = "0" # 댓글을 막아둔 경우
                
            # 결과 저장
            data_list.append({
                "아이디": blogger_id,
                "날짜": post_date,
                "본문 텍스트": post_text,
                "좋아요 수": like_count,
                "댓글 수": comment_count,
                "태그": tags
            })
            
            print(f"[{idx}/{len(post_links)}] 수집 완료 - 아이디: {blogger_id}")
            
        except Exception as e:
            print(f"[{idx}/{len(post_links)}] 수집 에러 발생 ({link}): {e}")
            continue
            
    # 브라우저 종료
    driver.quit()
    
    # 3. 데이터프레임 변환 및 엑셀(CSV) 저장
    df = pd.DataFrame(data_list)
    return df

if __name__ == "__main__":
    # 검색할 키워드와 수집할 최대 게시글 수 설정
    KEYWORD = "맛집" # 원하는 검색어로 변경하세요
    MAX_POSTS = 10   # 테스트용으로 10개 설정. 필요에 따라 늘리세요.
    
    result_df = crawl_naver_blog(keyword=KEYWORD, max_posts=MAX_POSTS)
    
    # CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
    file_name = f"naver_blog_reviews_{KEYWORD}.csv"
    result_df.to_csv(file_name, index=False, encoding='utf-8-sig')
    
    print(f"\n수집 완료! '{file_name}' 파일로 저장되었습니다.")

게시글 링크를 수집 중입니다...


KeyboardInterrupt: 